# Решение задачи кредитного скоринга 
Решение задачи кредитного скоринга будет представлено в нескольких ноутбуках. В данном ноутбуке строятся признаки заявки по выводам EDA, проверяется вклад каждой группы признаков и сохраняется итоговый сагрегированный датасет.

## Данные
Датасет соревнования устроен таким образом, что кредиты для тренировочной выборки взяты за период в М месяцев, а кредиты для тестовой выборки взяты за последующие K месяцев.

Каждая запись кредитной истории содержит самую разнообразную информацию о прошлом кредите клиента, например, сумму, отношение клиента к кредиту, дату открытия и закрытия, информацию о просрочках по платежам и др. Все публикуемые данные тщательно анонимизированы.

Целевая переменная – бинарная величина, принимающая значения 0 и 1, где 1 соответствует дефолту клиента по кредиту.

## Проверка решения
Итоговая метрика решения - ROC AUC. Решения принимаются в виде csv-файла с двумя колонками: “id” – уникальный идентификатор и “score” – предсказание дефолта клиента по кредиту.

## Импорт библиотек

In [1]:
# Импорт библиотек
import json
import sys
import time

sys.path.append('../src')

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

from credit_scoring.features.aggregations import build_features
from credit_scoring.features.categories import bin_share_features, category_features
from credit_scoring.features.payments import payment_features
from credit_scoring.features.recent import recent_loan_features
from credit_scoring.utils.read import read_features, read_target
from credit_scoring.validation import time_split

pd.set_option('display.max_rows', 100)
RANDOM_STATE = 13

## Группы признаков
По выводам EDA к базовым агрегатам (mean / max / sum / std, mode / nunique) добавляем группы признаков. Код каждой группы лежит в src/credit_scoring/features/:

| Группа | Модуль | Что считает | Вывод EDA |
|---|---|---|---|
| `base` | aggregations.py | прежние 149 агрегатов | - |
| `payments` | payments.py | доли статусов платежей в окнах 1/3/6/12/25 месяцев, доля просрочек среди месяцев с данными, месяцев с последней просрочки; кодировка `enc_paym_11/20/24` исправлена | коды статусов не упорядочены по риску, свежие месяцы важнее, в трёх месяцах сдвиг кодировки |
| `categories` | categories.py | доли и количества кредитов каждого значения категорий | редкие значения (`credit_status = 5`) очень рискованные, mode их теряет |
| `bins` | categories.py | доли кредитов в каждом бине для 5 важных `pre_` признаков | номера бинов не упорядочены по риску |
| `last` / `first` | recent.py | признаки крайнего кредита и средние по 3 крайним кредитам с конца / начала по `rn` | свежие кредиты должны быть информативнее; направление `rn` по времени не установлено, поэтому проверяем оба |

Все функции не используют таргет, поэтому одинаково применяются к train и test, и утечки нет.

## Построение признаков

In [2]:
FEATURE_GROUPS = {
    'base': build_features,
    'payments': payment_features,
    'categories': category_features,
    'bins': bin_share_features,
    'last': lambda df: recent_loan_features(df, n_loans=3, from_end=True),
    'first': lambda df: recent_loan_features(df, n_loans=3, from_end=False),
}

def build_all_features(data_path):
    """Строит все группы признаков для выборки; возвращает словарь группа -> таблица признаков (индекс - id).

    Группы не склеиваем в одну таблицу: вместе это ~400 колонок на 3 млн заявок, и склейка
    удвоила бы потребление памяти. Нужные колонки собираются под каждый эксперимент.
    """
    loans = read_features(data_path)
    parts = {}
    for name, fn in FEATURE_GROUPS.items():
        start = time.time()
        parts[name] = fn(loans).astype('float32')
        print(f"{name:10s}: {parts[name].shape[1]:3d} признаков, {time.time() - start:5.1f} s")
    del loans
    # Все группы агрегированы groupby по id, поэтому индексы должны совпадать
    ids = parts['base'].index
    assert ids.is_monotonic_increasing and all(p.index.equals(ids) for p in parts.values())
    return parts

In [3]:
# Признаки train
train_parts = build_all_features('../raw_datasets/train_data')
ids = train_parts['base'].index
target = read_target('../raw_datasets/train_target.csv')['flag'].reindex(ids)
assert target.notna().all()

groups = {name: list(part.columns) for name, part in train_parts.items()}
col_group = {c: name for name, cols in groups.items() for c in cols}
print(f"Память под признаки: {sum(p.memory_usage().sum() for p in train_parts.values()) / 2**30:.2f} GB")

base      : 149 признаков,  25.8 s


payments  :  55 признаков,  18.1 s


categories:  52 признаков,   2.9 s


bins      :  79 признаков,   6.3 s


last      :  39 признаков,   6.4 s


first     :  39 признаков,   5.7 s
Память под признаки: 4.66 GB


In [4]:
# Проверим, что названия колонок в группах не пересекаются
all_cols = [c for cols in groups.values() for c in cols]
assert len(all_cols) == len(set(all_cols)), 'дубли колонок между группами'
pd.Series({name: len(cols) for name, cols in groups.items()}, name='n_features')

base          149
payments       55
categories     52
bins           79
last           39
first          39
Name: n_features, dtype: int64

In [5]:
def get_X(cols, rows=slice(None)):
    """Собирает из групп таблицу с колонками cols и строками rows (позиции в порядке id)."""
    by_group = {}
    for c in cols:
        by_group.setdefault(col_group[c], []).append(c)
    return pd.concat([train_parts[g].iloc[rows][cs] for g, cs in by_group.items()], axis=1)

## Вклад групп признаков
Будем добавлять группы по одной к базовому набору и смотреть на ROC-AUC на valid - на той же схеме валидации по времени, что и в остальных ноутбуках. oot в этом ноутбуке не используется вообще.

Модель для сравнения - XGBoost с параметрами деревьев из тюнинга, но с `learning_rate = 0.1` и early stopping по valid: так каждое обучение занимает пару минут, а сравнение между наборами признаков остаётся честным (все наборы учатся одной и той же моделью).

Правило: группа остаётся, если улучшает valid ROC-AUC хотя бы на 0.0005. Все наборы оцениваются на одной valid выборке, поэтому разница между ними сравнима.

In [6]:
# Разбиение по времени (то же, что в остальных ноутбуках); oot дальше не используем
train_df, valid_df, _ = time_split(target.to_frame())
n_train, n_valid = len(train_df), len(valid_df)
TRAIN_ROWS, VALID_ROWS = slice(0, n_train), slice(n_train, n_train + n_valid)
del train_df, valid_df

with open('../artifacts/best_params.json', 'r', encoding='utf-8') as f:
    tree_params = {k: v for k, v in json.load(f).items() if k not in ('learning_rate', 'n_estimators')}
tree_params

{'max_depth': 5,
 'min_child_weight': 1.6028310982259062,
 'subsample': 0.8813688233629731,
 'colsample_bytree': 0.7488629804880247,
 'reg_lambda': 10.49660026127523,
 'reg_alpha': 3.0004452380967637}

In [7]:
y_train = target.iloc[TRAIN_ROWS]
y_valid = target.iloc[VALID_ROWS]

def evaluate(cols):
    """Valid ROC-AUC модели на наборе признаков cols (обучение на train, early stopping по valid)."""
    X_tr = get_X(cols, TRAIN_ROWS)
    X_va = get_X(cols, VALID_ROWS)
    model = XGBClassifier(
        **tree_params, learning_rate=0.1, n_estimators=3000, early_stopping_rounds=100,
        eval_metric='auc', tree_method='hist', random_state=RANDOM_STATE,
    )
    model.fit(X_tr, y_train, eval_set=[(X_va, y_valid)], verbose=False)
    return model.best_score, model

results = []
def run(name, cols):
    start = time.time()
    auc, model = evaluate(cols)
    results.append({'experiment': name, 'n_features': len(cols), 'valid_roc_auc': auc, 'n_trees': model.best_iteration + 1})
    print(f"{name:45s} {len(cols):4d} признаков  valid ROC-AUC = {auc:.4f}  ({time.time() - start:.0f} s)")
    return auc, model

In [8]:
# Базовый набор
base_cols = groups['base']
base_auc, _ = run('base', base_cols)

base                                           149 признаков  valid ROC-AUC = 0.7279  (155 s)


### Статусы платежей
Сравним два варианта: добавить доли статусов к базовым агрегатам или заменить ими прежние `enc_paym_*_mean/max` (25 месяцев x 2 агрегата = 50 признаков со «средним кодом» статуса).

In [9]:
# payments: добавить или заменить средние коды статусов
old_paym_cols = [c for c in base_cols if c.startswith('enc_paym_')]
add_auc, _ = run('base + payments', base_cols + groups['payments'])
replace_cols = [c for c in base_cols if c not in old_paym_cols] + groups['payments']
replace_auc, _ = run('base - enc_paym_*_mean/max + payments', replace_cols)

base + payments                                204 признаков  valid ROC-AUC = 0.7288  (214 s)


base - enc_paym_*_mean/max + payments          154 признаков  valid ROC-AUC = 0.7293  (241 s)


In [10]:
# Жадно добавляем остальные группы, начиная с лучшего варианта payments
TOL = 0.0005
current_cols, current_auc = base_cols, base_auc
if max(add_auc, replace_auc) >= base_auc + TOL:
    # При близком качестве предпочитаем замену - она даёт меньше признаков
    if replace_auc >= add_auc - TOL:
        current_cols, current_auc = replace_cols, replace_auc
    else:
        current_cols, current_auc = base_cols + groups['payments'], add_auc
print(f"Текущий набор: {len(current_cols)} признаков, valid ROC-AUC = {current_auc:.4f}")

decisions = {}
for group in ['categories', 'bins']:
    auc, _ = run(f'current + {group}', current_cols + groups[group])
    decisions[group] = auc >= current_auc + TOL
    if decisions[group]:
        current_cols, current_auc = current_cols + groups[group], auc
print(f"Текущий набор: {len(current_cols)} признаков, valid ROC-AUC = {current_auc:.4f}")

Текущий набор: 154 признаков, valid ROC-AUC = 0.7293


current + categories                           206 признаков  valid ROC-AUC = 0.7340  (285 s)


current + bins                                 285 признаков  valid ROC-AUC = 0.7458  (320 s)
Текущий набор: 285 признаков, valid ROC-AUC = 0.7458


### Крайние кредиты
Проверим признаки крайних кредитов с обоих концов по `rn`. Если `rn` идёт от старых кредитов к новым, группа `last` должна давать больший прирост, чем `first`.

In [11]:
# last vs first
last_auc, _ = run('current + last', current_cols + groups['last'])
first_auc, _ = run('current + first', current_cols + groups['first'])
best_recent = 'last' if last_auc >= first_auc else 'first'
best_recent_auc = max(last_auc, first_auc)
decisions['recent'] = best_recent_auc >= current_auc + TOL
if decisions['recent']:
    current_cols, current_auc = current_cols + groups[best_recent], best_recent_auc
print(f"Текущий набор: {len(current_cols)} признаков, valid ROC-AUC = {current_auc:.4f}")

current + last                                 324 признаков  valid ROC-AUC = 0.7553  (377 s)


current + first                                324 признаков  valid ROC-AUC = 0.7484  (435 s)
Текущий набор: 324 признаков, valid ROC-AUC = 0.7553


In [12]:
# Сводка экспериментов
results_df = pd.DataFrame(results)
results_df['gain_vs_base'] = results_df['valid_roc_auc'] - base_auc
results_df.round(4)

,experiment,n_features,valid_roc_auc,n_trees,gain_vs_base
0,base,149,0.7279,676,0.0000
1,base + payments,204,0.7288,673,0.0009
2,base - enc_paym_*_mean/max + payments,154,0.7293,919,0.0014
3,current + categories,206,0.7340,837,0.0061
4,current + bins,285,0.7458,685,0.0179
5,current + last,324,0.7553,695,0.0274
6,current + first,324,0.7484,815,0.0206


Выводы:
- **payments**: замена 50 признаков «среднего кода статуса» (`enc_paym_*_mean/max`) на доли статусов в окнах даёт +0.0014 при почти том же числе признаков (154 против 149), а простое добавление - меньше (+0.0009). Доли статусов полностью заменяют прежние агрегаты, и лишние скоррелированные признаки только мешают;
- **categories**: доли и количества значений категорий дают +0.005 - модель использует информацию о редких рискованных значениях, которую mode терял;
- **bins**: самый большой прирост, +0.012. Подтверждается вывод EDA: номера бинов не упорядочены по риску, и mean по номеру бина терял рискованные бины;
- **last vs first**: признаки последних кредитов (наибольший `rn`) дают +0.0095, первых - только +0.0026. Это экспериментально подтверждает, что `rn` идёт от старых кредитов к новым, и что свежие кредиты информативнее. Берём `last`;
- в сумме valid ROC-AUC вырос с 0.728 до 0.755 (+0.027) - в три раза больше, чем дал тюнинг. Качество признаков важнее подбора параметров.

## Проверка дрейфа
В EDA train и test различались с adversarial ROC-AUC 0.79. Посмотрим, какие признаки итогового набора сильнее всего отличают ранние заявки train (первые 20% по `id`) от поздних (последние 20%), и проверим, не лучше ли без самых дрейфующих из них. Используем ранние и поздние заявки train, а не test, - решения принимаются только на данных с таргетом.

In [13]:
# Adversarial validation: ранние vs поздние заявки train
n = len(target)
early = get_X(current_cols, slice(0, n // 5)).sample(200_000, random_state=RANDOM_STATE)
late = get_X(current_cols, slice(n - n // 5, n)).sample(200_000, random_state=RANDOM_STATE)
X_adv = pd.concat([early, late])
y_adv = np.r_[np.zeros(len(early)), np.ones(len(late))]
del early, late

adv_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, tree_method='hist', random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
adv_auc = cross_val_score(adv_model, X_adv, y_adv, cv=cv, scoring='roc_auc').mean()
adv_model.fit(X_adv, y_adv)
del X_adv
drift_importance = pd.Series(adv_model.feature_importances_, index=adv_model.feature_names_in_).sort_values(ascending=False)
print(f"Adversarial ROC-AUC (ранние vs поздние): {adv_auc:.4f}")
drift_importance.head(15)

Adversarial ROC-AUC (ранние vs поздние): 0.9240


enc_loans_credit_type_1_share            0.226114
enc_loans_account_holder_type_4_share    0.077555
enc_loans_credit_type_7_share            0.055236
pre_since_confirmed_max                  0.044645
pre_loans_credit_cost_rate_max           0.029517
enc_loans_credit_type_6_share            0.018674
enc_loans_account_holder_type_4_count    0.016921
pre_maxover2limit_3_share                0.016386
pre_loans_credit_cost_rate_2_share       0.015793
paym_last3_s3_share_mean                 0.011800
pre_loans_credit_cost_rate_11_share      0.011561
pre_loans_credit_cost_rate_0_share       0.011083
is_zero_loans6090_mean                   0.010961
paym_last3_s3_share_max                  0.010244
pre_since_opened_max                     0.009811
dtype: float32

In [14]:
# Пробуем убрать самые дрейфующие признаки: топ-5 и топ-10
for top in [5, 10]:
    drifting = drift_importance.index[:top].tolist()
    auc, _ = run(f'current - top-{top} drifting', [c for c in current_cols if c not in drifting])
    # Убираем, если качество не падает: при прочих равных модель без дрейфующих признаков устойчивее
    if auc >= current_auc - 0.0002:
        current_cols = [c for c in current_cols if c not in drifting]
        current_auc = auc
        print(f"  убраны: {drifting}")
print(f"Итоговый набор: {len(current_cols)} признаков, valid ROC-AUC = {current_auc:.4f}")

current - top-5 drifting                       319 признаков  valid ROC-AUC = 0.7552  (437 s)
  убраны: ['enc_loans_credit_type_1_share', 'enc_loans_account_holder_type_4_share', 'enc_loans_credit_type_7_share', 'pre_since_confirmed_max', 'pre_loans_credit_cost_rate_max']


current - top-10 drifting                      314 признаков  valid ROC-AUC = 0.7551  (378 s)
  убраны: ['enc_loans_credit_type_1_share', 'enc_loans_account_holder_type_4_share', 'enc_loans_credit_type_7_share', 'pre_since_confirmed_max', 'pre_loans_credit_cost_rate_max', 'enc_loans_credit_type_6_share', 'enc_loans_account_holder_type_4_count', 'pre_maxover2limit_3_share', 'pre_loans_credit_cost_rate_2_share', 'paym_last3_s3_share_mean']
Итоговый набор: 314 признаков, valid ROC-AUC = 0.7551


Выводы:
- с новыми признаками ранние и поздние заявки train различаются гораздо сильнее: adversarial ROC-AUC 0.92 против 0.80 на прежних признаках. Сильнее всего во времени меняются доли типов кредита (`enc_loans_credit_type_*_share`) и типов отношения к кредиту - видимо, со временем меняется структура кредитных продуктов;
- удаление 10 самых дрейфующих признаков почти не меняет valid ROC-AUC (0.7553 → 0.7551, в пределах допуска 0.0002 на каждом шаге), поэтому убираем их: при равном качестве модель без дрейфующих признаков должна быть устойчивее на будущих данных;
- остальные новые признаки тоже дрейфуют, поэтому прирост на valid обязательно нужно подтвердить на oot - это сделаем в конце, после пересчёта отбора признаков, тюнинга и калибровки.

## Сохранение
Сохраним итоговый набор признаков для train (вместе с таргетом) и test - в те же файлы, что читают следующие ноутбуки. Признаки храним во float32: XGBoost всё равно работает с ними во float32.

In [15]:
# Итоговая таблица train
final_cols = current_cols
pd.concat([target, get_X(final_cols)], axis=1)[['flag'] + final_cols].to_parquet('../artifacts/aggregated_df.parquet')
del train_parts

# Те же признаки для test - теми же функциями
test_parts = build_all_features('../raw_datasets/test_data')
test_features = pd.concat([test_parts[g][[c for c in cols if c in final_cols]] for g, cols in groups.items()], axis=1)
test_features[final_cols].to_parquet('../artifacts/test_features.parquet')
del test_parts

# Состав групп - для справки и для пайплайна в src
with open('../artifacts/feature_groups.json', 'w', encoding='utf-8') as f:
    json.dump({name: [c for c in cols if c in final_cols] for name, cols in groups.items()}, f, ensure_ascii=False, indent=4)
print(f"train: {len(target):,} x {len(final_cols)}, test: {len(test_features):,} x {len(final_cols)}")

base      : 149 признаков,   4.0 s


payments  :  55 признаков,   2.7 s


categories:  52 признаков,   0.5 s


bins      :  79 признаков,   0.8 s


last      :  39 признаков,   0.8 s


first     :  39 признаков,   0.7 s


train: 3,000,000 x 314, test: 500,000 x 314


## Итоги
1. Добавлены группы признаков по выводам EDA (код в src/credit_scoring/features/): доли статусов платежей с исправленной кодировкой, доли категорий, доли бинов, признаки последних кредитов.
2. Прирост valid ROC-AUC: 0.728 → 0.755. Самые полезные группы - доли бинов и последние кредиты.
3. Прежние `enc_paym_*_mean/max` заменены долями статусов, 10 самых дрейфующих признаков удалены. Итого 314 признаков.
4. Итоговые train и test сохранены в `artifacts/aggregated_df.parquet` и `artifacts/test_features.parquet`, состав групп - в `artifacts/feature_groups.json`.

Далее - отбор признаков, тюнинг и калибровка на новом наборе, затем проверка прироста на oot.